In [1]:
!pip -q install pyarrow pandas numpy

In [4]:
from google.colab import files

# Upload these three:
# - 02_zero_inflation_augmented.parquet
# - 02_zero_mask.npy
# - 02_zero_confidence.npy
uploaded = files.upload()

print("Uploaded:", list(uploaded.keys()))


Saving 02_zero_confidence.npy to 02_zero_confidence (2).npy
Saving 02_zero_inflation_augmented.parquet to 02_zero_inflation_augmented (1).parquet
Saving 02_zero_mask.npy to 02_zero_mask (1).npy
Uploaded: ['02_zero_confidence (2).npy', '02_zero_inflation_augmented (1).parquet', '02_zero_mask (1).npy']


In [5]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

IN_PARQUET = "02_zero_inflation_augmented.parquet"

df = pq.read_table(IN_PARQUET).to_pandas()
print("Loaded parquet:", df.shape)
print("Columns:", list(df.columns))

# Optional cross-check with .npy files (recommended once, then you can skip)
if "02_zero_mask.npy" in uploaded and "02_zero_confidence.npy" in uploaded:
    mask_npy = np.load("02_zero_mask.npy")
    conf_npy = np.load("02_zero_confidence.npy")

    mask_col = df["zero_mask"].to_numpy()
    conf_col = df["zero_confidence"].to_numpy()

    # Cast for stable comparisons
    ok_mask = np.array_equal(mask_npy.astype(mask_col.dtype, copy=False), mask_col)
    ok_conf = np.allclose(conf_npy.astype(conf_col.dtype, copy=False), conf_col, rtol=1e-6, atol=1e-6)

    print("Mask matches parquet:", ok_mask)
    print("Confidence matches parquet:", ok_conf)


Loaded parquet: (301340, 19)
Columns: ['Time', 'i_value', 'j_value', 'latitude', 'longitude', 'u_west_to_east_wind', 'v_south_to_north_wind', 'temprature', 'relative_humidity', 'vertical_velocity', 'pressure', 'water_vapour', 'turbulent_kinatic_energy', 'precipitation_rate', 'sensible_heat_flux', 'Latent_heat_flux', 'tracer_concentration', 'zero_mask', 'zero_confidence']


In [6]:
import json
from typing import Dict, Tuple

def _col(df: pd.DataFrame, name: str, *, alt=(), default=None) -> str:
    """
    Return the first matching column among [name] + alt.
    If none found and default is not None, create a default column and return its name.
    """
    candidates = [name] + list(alt)
    for c in candidates:
        if c in df.columns:
            return c
    if default is not None:
        df[name] = default
        return name
    raise KeyError(f"Missing required column(s): {candidates}")

def dew_point_c_magnus(temp_c: np.ndarray, rh_pct: np.ndarray) -> np.ndarray:
    """
    Dew point approximation (Magnus formula). Vectorized.
    temp_c: Celsius
    rh_pct: percent (0–100)
    """
    a, b = 17.27, 237.7
    rh = np.clip(rh_pct / 100.0, 1e-6, 1.0)
    gamma = (a * temp_c) / (b + temp_c) + np.log(rh)
    return (b * gamma) / (a - gamma)


In [7]:
def build_env_context_features(df_in: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    df = df_in.copy()

    # ---- Resolve required columns (your current parquet uses these exact names) ----
    c_time = _col(df, "Time", alt=("time",), default=None)
    c_i = _col(df, "i_value")
    c_j = _col(df, "j_value")
    c_u = _col(df, "u_west_to_east_wind", alt=("u",))
    c_v = _col(df, "v_south_to_north_wind", alt=("v",))
    c_temp = _col(df, "temprature", alt=("temperature", "temp"))
    c_rh = _col(df, "relative_humidity", alt=("rh",))
    c_w = _col(df, "vertical_velocity", alt=("w",))
    c_p = _col(df, "pressure", alt=("press",))
    c_wv = _col(df, "water_vapour", alt=("water_vapor",))
    c_tke = _col(df, "turbulent_kinatic_energy", alt=("turbulent_kinetic_energy", "tke"))
    c_pr = _col(df, "precipitation_rate", alt=("precip_rate",))
    c_shf = _col(df, "sensible_heat_flux", alt=("shf",))
    c_lhf = _col(df, "Latent_heat_flux", alt=("latent_heat_flux", "lhf"))
    c_zconf = _col(df, "zero_confidence")
    c_zmask = _col(df, "zero_mask")

    # ---- Pull as arrays (float32 for memory + speed) ----
    eps = np.float32(1e-6)

    u = df[c_u].astype("float32").to_numpy()
    v = df[c_v].astype("float32").to_numpy()
    temp_c = df[c_temp].astype("float32").to_numpy()
    rh = df[c_rh].astype("float32").to_numpy()
    w = df[c_w].astype("float32").to_numpy()

    # pressure: your data often uses hPa; convert to Pa for density formula
    p_hpa = df[c_p].astype("float32").to_numpy()
    p_pa = (p_hpa * np.float32(100.0)).astype("float32")

    wv = df[c_wv].astype("float32").to_numpy()
    tke = df[c_tke].astype("float32").to_numpy()
    pr = df[c_pr].astype("float32").to_numpy()
    shf = df[c_shf].astype("float32").to_numpy()
    lhf = df[c_lhf].astype("float32").to_numpy()

    zero_conf = df[c_zconf].astype("float32").to_numpy()

    # ---- Wind context ----
    wind_speed = np.sqrt(u*u + v*v).astype("float32")
    wind_dir = np.arctan2(v, u).astype("float32")  # radians
    u_unit = (u / (wind_speed + eps)).astype("float32")
    v_unit = (v / (wind_speed + eps)).astype("float32")

    df["ctx_wind_speed"] = wind_speed
    df["ctx_wind_speed_sq"] = (wind_speed * wind_speed).astype("float32")
    df["ctx_wind_power"] = (wind_speed ** 3).astype("float32")
    df["ctx_wind_log1p"] = np.log1p(wind_speed).astype("float32")
    df["ctx_wind_dir_rad"] = wind_dir
    df["ctx_wind_dir_sin"] = np.sin(wind_dir).astype("float32")
    df["ctx_wind_dir_cos"] = np.cos(wind_dir).astype("float32")
    df["ctx_wind_u_unit"] = u_unit
    df["ctx_wind_v_unit"] = v_unit

    # ---- Precipitation / turbulence / vertical motion ----
    pr_pos = np.maximum(pr, 0).astype("float32")
    tke_pos = np.maximum(tke, 0).astype("float32")
    w_abs = np.abs(w).astype("float32")

    df["ctx_precip_flag"] = (pr_pos > 0).astype("int8")
    df["ctx_precip_log1p"] = np.log1p(pr_pos).astype("float32")
    df["ctx_tke_log1p"] = np.log1p(tke_pos).astype("float32")
    df["ctx_vertical_abs"] = w_abs
    df["ctx_vertical_log1p"] = np.log1p(w_abs).astype("float32")

    # turbulence intensity proxy: sqrt(2/3*TKE) / wind_speed
    turb_vel = np.sqrt(np.float32(2.0/3.0) * tke_pos).astype("float32")
    df["ctx_turb_intensity"] = (turb_vel / (wind_speed + eps)).astype("float32")

    # ---- Thermodynamic context ----
    temp_k = (temp_c + np.float32(273.15)).astype("float32")
    df["ctx_temp_k"] = temp_k
    df["ctx_humidity_frac"] = np.clip(rh / 100.0, 0, 1).astype("float32")
    df["ctx_dew_point_c"] = dew_point_c_magnus(temp_c, rh).astype("float32")

    # density = p / (R*T)
    df["ctx_air_density"] = (p_pa / (np.float32(287.05) * temp_k + eps)).astype("float32")

    df["ctx_water_vapour_log1p"] = np.log1p(np.maximum(wv, 0)).astype("float32")

    # heat flux proxies
    heat_total = (shf + lhf).astype("float32")
    df["ctx_heat_flux_total"] = heat_total
    df["ctx_bowen_ratio"] = (shf / (lhf + eps)).astype("float32")

    # ---- Confidence propagation (environment penalties) ----
    # Rationale: precipitation + turbulence + strong vertical motion usually degrade measurement reliability.
    env_conf = np.exp(
        -np.float32(0.90) * df["ctx_precip_log1p"].to_numpy(dtype="float32")
        -np.float32(0.35) * df["ctx_tke_log1p"].to_numpy(dtype="float32")
        -np.float32(0.05) * df["ctx_vertical_log1p"].to_numpy(dtype="float32")
    ).astype("float32")
    env_conf = np.clip(env_conf, 0.0, 1.0).astype("float32")

    overall_conf = np.clip(zero_conf * env_conf, 0.0, 1.0).astype("float32")

    df["ctx_env_confidence"] = env_conf
    df["ctx_overall_confidence"] = overall_conf

    # ---- Spatial normalization ----
    i = df[c_i].to_numpy()
    j = df[c_j].to_numpy()
    i_min, i_max = int(np.min(i)), int(np.max(i))
    j_min, j_max = int(np.min(j)), int(np.max(j))

    df["ctx_i_norm"] = ((i - i_min) / max(1, (i_max - i_min))).astype("float32")
    df["ctx_j_norm"] = ((j - j_min) / max(1, (j_max - j_min))).astype("float32")

    # ---- Time features (safe if constant; useful for later spike encoding) ----
    # Cyclic features over day. If Time parsing fails, fall back to zeros.
    try:
        t = pd.to_datetime(df[c_time], errors="coerce")
        if t.notna().any():
            t0 = t.min()
            dt_s = (t - t0).dt.total_seconds().astype("float32").to_numpy()
            day_s = np.float32(24 * 3600)
            phase = (2 * np.pi * ((dt_s % day_s) / day_s)).astype("float32")
            df["ctx_dt_seconds"] = dt_s
            df["ctx_time_sin"] = np.sin(phase).astype("float32")
            df["ctx_time_cos"] = np.cos(phase).astype("float32")
        else:
            df["ctx_dt_seconds"] = np.zeros(len(df), dtype="float32")
            df["ctx_time_sin"] = np.zeros(len(df), dtype="float32")
            df["ctx_time_cos"] = np.zeros(len(df), dtype="float32")
            t0 = None
    except Exception:
        df["ctx_dt_seconds"] = np.zeros(len(df), dtype="float32")
        df["ctx_time_sin"] = np.zeros(len(df), dtype="float32")
        df["ctx_time_cos"] = np.zeros(len(df), dtype="float32")
        t0 = None

    ctx_cols = [c for c in df.columns if c.startswith("ctx_")]

    spec = {
        "stage": "04_environment_context_features",
        "rows": int(len(df)),
        "ctx_feature_count": int(len(ctx_cols)),
        "ctx_features": ctx_cols,
        "spatial_norm": {"i_min": i_min, "i_max": i_max, "j_min": j_min, "j_max": j_max},
        "confidence": {
            "env_confidence": "exp(-0.90*log1p(precip) - 0.35*log1p(tke) - 0.05*log1p(|vertical|))",
            "overall_confidence": "zero_confidence * env_confidence",
        },
        "note": "Traffic/terrain are not present in this dataset; join external GIS layers later if available.",
    }

    return df, spec

df04, spec04 = build_env_context_features(df)
print("Stage 04 created:", df04.shape, "ctx_*:", spec04["ctx_feature_count"])


Stage 04 created: (301340, 48) ctx_*: 29


In [8]:
import pyarrow as pa
import pyarrow.parquet as pq
import json

OUT_PARQUET = "04_env_context_features.parquet"
OUT_SPEC = "04_env_context_features_spec.json"

# Parquet is best for size; zstd is usually available in Colab.
pq.write_table(pa.Table.from_pandas(df04, preserve_index=False), OUT_PARQUET, compression="zstd")

with open(OUT_SPEC, "w") as f:
    json.dump(spec04, f, indent=2)

# Optional: export just the ctx feature matrix for neuromorphic encoding
ctx_cols = spec04["ctx_features"]
X_ctx = df04[ctx_cols].astype("float32").to_numpy()
np.save("04_ctx_matrix.npy", X_ctx)
with open("04_ctx_columns.json", "w") as f:
    json.dump(ctx_cols, f, indent=2)

# Optional: save the propagated confidence alone
np.save("04_ctx_overall_confidence.npy", df04["ctx_overall_confidence"].astype("float32").to_numpy())

print("Saved:", OUT_PARQUET, OUT_SPEC, "04_ctx_matrix.npy", "04_ctx_columns.json", "04_ctx_overall_confidence.npy")


Saved: 04_env_context_features.parquet 04_env_context_features_spec.json 04_ctx_matrix.npy 04_ctx_columns.json 04_ctx_overall_confidence.npy


In [9]:
def pct(x):
    return float(100.0 * x)

zmask = df04["zero_mask"].to_numpy()
wind = df04["ctx_wind_speed"].to_numpy()
conf = df04["ctx_overall_confidence"].to_numpy()

print("Rows:", len(df04))
print("Zero-inflated fraction:", f"{pct(np.mean(zmask==1)):.2f}%")
print("Non-zero fraction:", f"{pct(np.mean(zmask==0)):.2f}%")

for name, arr in [("wind_speed", wind), ("overall_confidence", conf)]:
    p10, p50, p90 = np.percentile(arr, [10, 50, 90])
    print(f"{name} p10/p50/p90:", float(p10), float(p50), float(p90))

print("\nctx_* features created:", len([c for c in df04.columns if c.startswith("ctx_")]))


Rows: 301340
Zero-inflated fraction: 96.43%
Non-zero fraction: 3.57%
wind_speed p10/p50/p90: 4.0826917648315435 5.46304988861084 6.716703701019288
overall_confidence p10/p50/p90: 0.5664109766483307 0.6222851276397705 0.6919226884841919

ctx_* features created: 29


In [10]:
# QSNN Methane Drone Pipeline — Stage 04 (Full): Environment & Context Features
# Permission: You may use, modify, and distribute this code freely.

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import json

def _col(df: pd.DataFrame, name: str, *, alt=(), default=None) -> str:
    """Return first matching column among name + alt. Optionally create default."""
    for c in [name] + list(alt):
        if c in df.columns:
            return c
    if default is not None:
        df[name] = default
        return name
    raise KeyError(f"Missing required column(s): {[name] + list(alt)}")

def dew_point_c_magnus(temp_c: np.ndarray, rh_pct: np.ndarray) -> np.ndarray:
    """Magnus dew-point approximation (vectorized)."""
    a, b = 17.27, 237.7
    rh = np.clip(rh_pct / 100.0, 1e-6, 1.0)
    gamma = (a * temp_c) / (b + temp_c) + np.log(rh)
    return (b * gamma) / (a - gamma)

def build_env_context_features_full(df_in: pd.DataFrame):
    df = df_in.copy()
    eps = np.float32(1e-6)

    # --- Resolve columns (robust to your file’s spellings) ---
    c_time = _col(df, "Time", alt=("time",), default=None)
    c_i    = _col(df, "i_value")
    c_j    = _col(df, "j_value")
    c_u    = _col(df, "u_west_to_east_wind", alt=("u",))
    c_v    = _col(df, "v_south_to_north_wind", alt=("v",))
    c_temp = _col(df, "temprature", alt=("temperature", "temp"))
    c_rh   = _col(df, "relative_humidity", alt=("rh",))
    c_w    = _col(df, "vertical_velocity", alt=("w",))
    c_p    = _col(df, "pressure", alt=("press",))
    c_wv   = _col(df, "water_vapour", alt=("water_vapor",))
    c_tke  = _col(df, "turbulent_kinatic_energy", alt=("turbulent_kinetic_energy", "tke"))
    c_pr   = _col(df, "precipitation_rate", alt=("precip_rate",))
    c_shf  = _col(df, "sensible_heat_flux", alt=("shf",))
    c_lhf  = _col(df, "Latent_heat_flux", alt=("latent_heat_flux", "lhf"))
    c_zc   = _col(df, "zero_confidence")
    c_zm   = _col(df, "zero_mask")

    # --- Arrays ---
    u = df[c_u].astype("float32").to_numpy()
    v = df[c_v].astype("float32").to_numpy()
    temp_c = df[c_temp].astype("float32").to_numpy()
    rh = df[c_rh].astype("float32").to_numpy()
    w = df[c_w].astype("float32").to_numpy()

    p_hpa = df[c_p].astype("float32").to_numpy()            # assume hPa
    p_pa  = (p_hpa * np.float32(100.0)).astype("float32")   # Pa

    wv  = df[c_wv].astype("float32").to_numpy()
    tke = df[c_tke].astype("float32").to_numpy()
    pr  = df[c_pr].astype("float32").to_numpy()
    shf = df[c_shf].astype("float32").to_numpy()
    lhf = df[c_lhf].astype("float32").to_numpy()

    zero_conf = df[c_zc].astype("float32").to_numpy()

    # --- Wind context ---
    wind_speed = np.sqrt(u*u + v*v).astype("float32")
    wind_dir = np.arctan2(v, u).astype("float32")  # radians
    u_unit = (u / (wind_speed + eps)).astype("float32")
    v_unit = (v / (wind_speed + eps)).astype("float32")

    df["ctx_wind_speed"] = wind_speed
    df["ctx_wind_speed_sq"] = (wind_speed * wind_speed).astype("float32")
    df["ctx_wind_power"] = (wind_speed ** 3).astype("float32")
    df["ctx_wind_log1p"] = np.log1p(wind_speed).astype("float32")
    df["ctx_wind_dir_rad"] = wind_dir
    df["ctx_wind_dir_sin"] = np.sin(wind_dir).astype("float32")
    df["ctx_wind_dir_cos"] = np.cos(wind_dir).astype("float32")
    df["ctx_wind_u_unit"] = u_unit
    df["ctx_wind_v_unit"] = v_unit

    # --- Precip / turbulence / vertical ---
    pr_pos  = np.maximum(pr, 0).astype("float32")
    tke_pos = np.maximum(tke, 0).astype("float32")
    w_abs   = np.abs(w).astype("float32")

    df["ctx_precip_flag"] = (pr_pos > 0).astype("int8")
    df["ctx_precip_log1p"] = np.log1p(pr_pos).astype("float32")
    df["ctx_tke_log1p"] = np.log1p(tke_pos).astype("float32")
    df["ctx_vertical_abs"] = w_abs
    df["ctx_vertical_log1p"] = np.log1p(w_abs).astype("float32")

    turb_vel = np.sqrt(np.float32(2.0/3.0) * tke_pos).astype("float32")
    df["ctx_turb_intensity"] = (turb_vel / (wind_speed + eps)).astype("float32")

    # --- Thermodynamics ---
    temp_k = (temp_c + np.float32(273.15)).astype("float32")
    df["ctx_temp_k"] = temp_k
    df["ctx_humidity_frac"] = np.clip(rh / 100.0, 0, 1).astype("float32")
    df["ctx_dew_point_c"] = dew_point_c_magnus(temp_c, rh).astype("float32")
    df["ctx_air_density"] = (p_pa / (np.float32(287.05) * temp_k + eps)).astype("float32")
    df["ctx_water_vapour_log1p"] = np.log1p(np.maximum(wv, 0)).astype("float32")

    heat_total = (shf + lhf).astype("float32")
    df["ctx_heat_flux_total"] = heat_total
    df["ctx_bowen_ratio"] = (shf / (lhf + eps)).astype("float32")

    # --- Confidence propagation (environment penalties) ---
    env_conf = np.exp(
        -np.float32(0.90) * df["ctx_precip_log1p"].to_numpy(dtype="float32")
        -np.float32(0.35) * df["ctx_tke_log1p"].to_numpy(dtype="float32")
        -np.float32(0.05) * df["ctx_vertical_log1p"].to_numpy(dtype="float32")
    ).astype("float32")
    env_conf = np.clip(env_conf, 0.0, 1.0).astype("float32")
    df["ctx_env_confidence"] = env_conf
    df["ctx_overall_confidence"] = np.clip(zero_conf * env_conf, 0.0, 1.0).astype("float32")

    # --- Spatial normalization ---
    i = df[c_i].to_numpy()
    j = df[c_j].to_numpy()
    i_min, i_max = int(np.min(i)), int(np.max(i))
    j_min, j_max = int(np.min(j)), int(np.max(j))

    df["ctx_i_norm"] = ((i - i_min) / max(1, (i_max - i_min))).astype("float32")
    df["ctx_j_norm"] = ((j - j_min) / max(1, (j_max - j_min))).astype("float32")

    # --- Time cyclic features (safe even if Time is constant/missing) ---
    try:
        t = pd.to_datetime(df[c_time], errors="coerce")
        if t.notna().any():
            t0 = t.min()
            dt_s = (t - t0).dt.total_seconds().astype("float32").to_numpy()
            day_s = np.float32(24 * 3600)
            phase = (2 * np.pi * ((dt_s % day_s) / day_s)).astype("float32")
            df["ctx_dt_seconds"] = dt_s
            df["ctx_time_sin"] = np.sin(phase).astype("float32")
            df["ctx_time_cos"] = np.cos(phase).astype("float32")
        else:
            df["ctx_dt_seconds"] = np.zeros(len(df), dtype="float32")
            df["ctx_time_sin"] = np.zeros(len(df), dtype="float32")
            df["ctx_time_cos"] = np.zeros(len(df), dtype="float32")
    except Exception:
        df["ctx_dt_seconds"] = np.zeros(len(df), dtype="float32")
        df["ctx_time_sin"] = np.zeros(len(df), dtype="float32")
        df["ctx_time_cos"] = np.zeros(len(df), dtype="float32")

    ctx_cols = [c for c in df.columns if c.startswith("ctx_")]

    spec = {
        "stage": "04_environment_context_features_full",
        "rows": int(len(df)),
        "ctx_feature_count": int(len(ctx_cols)),
        "ctx_features": ctx_cols,
        "spatial_norm": {"i_min": i_min, "i_max": i_max, "j_min": j_min, "j_max": j_max},
        "confidence_formula": "overall = zero_confidence * exp(-0.90*log1p(precip) - 0.35*log1p(tke) - 0.05*log1p(|vertical|))",
        "note": "Traffic/terrain layers not present in base file; can be joined later from GIS sources."
    }
    return df, spec

# --- Load Stage-03 parquet (recommended baseline) ---
IN_PARQUET = "02_zero_inflation_augmented.parquet"
df03 = pq.read_table(IN_PARQUET).to_pandas()

df04_full, spec04_full = build_env_context_features_full(df03)

print("Rows:", len(df04_full))
print("ctx_* features created:", spec04_full["ctx_feature_count"])
print("First 10 ctx_* columns:", spec04_full["ctx_features"][:10])

# --- Save outputs ---
OUT_PARQUET = "04_env_context_features_full.parquet"
OUT_SPEC = "04_env_context_features_full_spec.json"
pq.write_table(pa.Table.from_pandas(df04_full, preserve_index=False), OUT_PARQUET, compression="zstd")
with open(OUT_SPEC, "w") as f:
    json.dump(spec04_full, f, indent=2)

print("Saved:", OUT_PARQUET, OUT_SPEC)


Rows: 301340
ctx_* features created: 29
First 10 ctx_* columns: ['ctx_wind_speed', 'ctx_wind_speed_sq', 'ctx_wind_power', 'ctx_wind_log1p', 'ctx_wind_dir_rad', 'ctx_wind_dir_sin', 'ctx_wind_dir_cos', 'ctx_wind_u_unit', 'ctx_wind_v_unit', 'ctx_precip_flag']
Saved: 04_env_context_features_full.parquet 04_env_context_features_full_spec.json


In [11]:
zmask = df04_full["zero_mask"].to_numpy()
wind = df04_full["ctx_wind_speed"].to_numpy()
conf = df04_full["ctx_overall_confidence"].to_numpy()

print("Rows:", len(df04_full))
print("Zero-inflated fraction:", f"{100*np.mean(zmask==1):.2f}%")
print("Non-zero fraction:", f"{100*np.mean(zmask==0):.2f}%")

for name, arr in [("wind_speed", wind), ("overall_confidence", conf)]:
    p10, p50, p90 = np.percentile(arr, [10, 50, 90])
    print(f"{name} p10/p50/p90:", float(p10), float(p50), float(p90))

print("\nctx_* features created:", len([c for c in df04_full.columns if c.startswith('ctx_')]))


Rows: 301340
Zero-inflated fraction: 96.43%
Non-zero fraction: 3.57%
wind_speed p10/p50/p90: 4.0826917648315435 5.46304988861084 6.716703701019288
overall_confidence p10/p50/p90: 0.5664109766483307 0.6222851276397705 0.6919226884841919

ctx_* features created: 29


In [12]:
import numpy as np
import json
import pyarrow as pa
import pyarrow.parquet as pq

OUT_PARQUET = "04_env_context_features.parquet"
OUT_SPEC = "04_env_context_features_spec.json"

ctx_cols = [c for c in df04_full.columns if c.startswith("ctx_")]

spec = {
    "stage": "04_environment_context_features",
    "rows": int(len(df04_full)),
    "ctx_feature_count": int(len(ctx_cols)),
    "ctx_features": ctx_cols,
    "recommended_next": "Stage 05 neuromorphic encoding (TTFS/Phase) using ctx_features + ctx_overall_confidence"
}

pq.write_table(pa.Table.from_pandas(df04_full, preserve_index=False), OUT_PARQUET, compression="zstd")
with open(OUT_SPEC, "w") as f:
    json.dump(spec, f, indent=2)

# Optional: export a compact matrix for spike encoding (fast, low RAM)
X_ctx = df04_full[ctx_cols].astype("float32").to_numpy()
np.save("04_ctx_matrix.npy", X_ctx)
np.save("04_ctx_overall_confidence.npy", df04_full["ctx_overall_confidence"].astype("float32").to_numpy())

with open("04_ctx_columns.json", "w") as f:
    json.dump(ctx_cols, f, indent=2)

print("Saved:", OUT_PARQUET, OUT_SPEC, "04_ctx_matrix.npy", "04_ctx_columns.json", "04_ctx_overall_confidence.npy")


Saved: 04_env_context_features.parquet 04_env_context_features_spec.json 04_ctx_matrix.npy 04_ctx_columns.json 04_ctx_overall_confidence.npy
